In [0]:
-- ================================================================================================================
-- VELIB NETWORK - GOLD LAYER
-- SQL ANALYTICS / BUSINESS QUESTIONS
-- ================================================================================================================
--
-- Purpose :
-- Use the curated Gold dimensional and fact tables to perform analytical SQL and answer business-oriented questions 
-- about bike availability, station capacity, and regional distribution across the Vélib network.
--
-- Dataset grain :
--   - gold_dim_commune: One record represents one unique geographical commune and its INSEE code.
--   - gold_dim_station: One record represents one physical bike station with static metadata and capacity.
--   - gol_fact_status: One record represents real-time operational status for one station at a given snapshot timestamp.
--
-- Main columns :
--   - gold_dim_commune: commune_name, code_insee
--   - gold_dim_station: station_code, station_name, capacity, latitude, longitude, commune_name, code_insee
--   - gol_fact_status: station_code, snapshot_time, is_installed, is_renting, is_returning, 
--                      bikes_available, mechanical_bikes, electric_bikes, docks_available, occupancy_rate
--
-- Assumed Gold table names :
-- gold.gold_dim_commune, gold.gold_dim_station, gold.gol_fact_status
--
-- Analytics objective :
-- Move from data preparation to business analysis: monitor network capacity, evaluate fleet composition 
-- (mechanical vs electric), assess station occupancy rates, detect critical stock-outs, and optimize rebalancing.
-- ================================================================================================================

-- Display the 3 tables 

SELECT * FROM workspace.gold.gold_velib_dim_commune;
SELECT * FROM workspace.gold.gold_velib_dim_station;
SELECT * FROM workspace.gold.gold_velib_fact_status;

-- Total Active Installed Stations and Overall Network Docking Capacity

SELECT 
    COUNT(DISTINCT station_name) as station_count,
    SUM(capacity) AS  network_capacity
FROM workspace.gold.gold_velib_dim_station;

-- Top 10 stations by Total Combined Station Docking Capacity

SELECT 
    station_name,
    capacity
FROM workspace.gold.gold_velib_dim_station
ORDER BY capacity DESC
LIMIT 10;

-- -- Top 10 commune by Total Combined Station Docking Capacity

SELECT 
    c.commune_name,
    SUM(s.capacity) AS total
FROM workspace.gold.gold_velib_dim_station s
JOIN workspace.gold.gold_velib_dim_commune c on s.code_insee=c.code_insee
GROUP BY c.commune_name
ORDER BY total DESC 
LIMIT 10;

-- Total Number and Location of Uninstalled or Inactive Stations 

SELECT 
    f.is_installed,
    f.station_code,
    s.station_name,
    s.capacity
FROM workspace.gold.gold_velib_fact_status f 
LEFT JOIN workspace.gold.gold_velib_dim_station s on f.station_code=s.station_code
WHERE f.is_installed ='false';

--   Average Station Docking Capacity across Each commune

SELECT 
    c.commune_name,
    ROUND (AVG (capacity),2) AS avg_capacity
FROM workspace.gold.gold_velib_dim_station s 
LEFT JOIN workspace.gold.gold_velib_dim_commune c on c.code_insee=s.code_insee
GROUP BY c.commune_name;

--  Total Real-Time Available Bikes versus Open  Docks 

SELECT 
    SUM (bikes_available) AS total_bikes,
    SUM (docks_available) AS total_docks
FROM workspace.gold.gold_velib_fact_status
WHERE is_renting = TRUE AND is_returning = TRUE;

-- Online Stations Where Both Renting and Returning Operations Are Disabled

SELECT DISTINCT
    s.station_code,
    s.station_name,
    c.commune_name
FROM workspace.gold.gold_velib_fact_status f
JOIN workspace.gold.gold_velib_dim_station s ON f.station_code = s.station_code
JOIN workspace.gold.gold_velib_dim_commune c ON s.code_insee = c.code_insee
WHERE f.is_renting = FALSE AND f.is_returning = FALSE;

--  Rank Communes by Absolute Station Density and Coverage

SELECT 
    c.commune_name,
    COUNT (s.station_name) AS total_stations ,
    RANK () OVER (ORDER BY COUNT(s.station_name) DESC ) AS ranking
FROM workspace.gold.gold_velib_dim_station s 
LEFT JOIN workspace.gold.gold_velib_dim_commune c on s.code_insee=c.code_insee
GROUP BY c.commune_name
ORDER BY total_stations DESC ;

--  Stations segmentation into Size Categories Based on Docking Capacity 

SELECT 
    station_code,
    station_name,
    capacity,
    CASE 
            WHEN capacity < 20 THEN 'small'
            WHEN capacity BETWEEN 20 AND 40 THEN 'medium'
            WHEN capacity >40 THEN 'Large'
            END AS station_size
FROM workspace.gold.gold_velib_dim_station ; 

--  Stations Encountering Full Capacity Depletion Blocking Returns

SELECT * 
FROM workspace.gold.gold_velib_fact_status;